<a href="https://colab.research.google.com/github/tmvenkadesh/AgenticAI/blob/main/Agentic_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh


downloading uv 0.10.8 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [ ]:
!uv pip install langchain-ollama langgraph langfuse pydantic python-dotenv --system


Using Python 3.12.12 environment at: /usr
Audited 5 packages in 81ms


In [ ]:
!apt-get install -y zstd


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (438 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"])
import time
time.sleep(3)  # wait for server to start


In [ ]:
!ollama pull llama3.2


In [ ]:
from google.colab import userdata
import os
os.environ["LANGFUSE_PUBLIC_KEY"] =userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"]=userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"]="https://cloud.langfuse.com"

In [ ]:


from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from typing import Annotated, TypedDict, Union, Literal
from langfuse.langchain import CallbackHandler
from pydantic import BaseModel, Field
from langfuse import get_client
from dotenv import load_dotenv
import os

load_dotenv()

langfuse = get_client()

langfuse_handler = CallbackHandler()

config ={
    "callbacks":[langfuse_handler]
}

class RouteDecision(BaseModel):
    """Classify the user query for routing."""
    category: Literal["TECHNICAL", "GENERAL"] = Field(
        description="The category of the question."
    )


class AgentState(TypedDict):
    question: str
    classification: str
    response: str

llm= ChatOllama(model="llama3.2",temperature=0)

structured_llm = llm.with_structured_output(RouteDecision)

def categorize_node(state:AgentState):
    print("***************** categorizing************")
    question=state["question"]

    system_prompt=("You are a router. Classify the input as 'TECHNICAL' or 'GENERAL'. "
        "Output ONLY the word 'TECHNICAL' or 'GENERAL'. "
        "Do not include explanations, punctuation, or conversational filler.")

    messages =[("system", system_prompt),
    ("human",question)]
   # response = llm.invoke(messages)
    response = structured_llm.invoke(question)

    print("response fromm llm :{0}", response.category)

    return {"classification": response.category.strip().upper()}

def technical_node(state: AgentState):
    print("--Handling Techical query--")
    return {"response": "Here is the some python code"}

def general_node(state:AgentState):
    print("Handling Genral query")
    return{"response": "here is your general questions answer"}

def decide_next_node(state:AgentState):
    category= state["classification"]

    if category == "TECHNICAL":
        return "technical"
    elif category == "GENERAL":
        return "general"
    else:
        return "unknown"

workflow = StateGraph(AgentState)

workflow.add_node("categorize", categorize_node)
workflow.add_node("technical",technical_node)
workflow.add_node("general",general_node)

workflow.set_entry_point("categorize")

workflow.add_conditional_edges(
    "categorize",
    decide_next_node,
    {
        "technical": "technical",
        "general": "general",
        "unknown": END
    }

)

workflow.add_edge("technical",END)
workflow.add_edge("general",END)

app = workflow.compile()

#Testing

#input = {"question": "write some python script to parse the json"}
input = {"question":"capital of india"}
app.invoke(input,config=config)



***************** categorizing************
response fromm llm :{0} GENERAL
Handling Genral query


{'question': 'capital of india',
 'classification': 'GENERAL',
 'response': 'here is your general questions answer'}